# Fase 4 — Entrenamiento y evaluación del detector YOLO

Este notebook implementa el flujo reproducible de entrenamiento y evaluación del detector YOLO para OptiSignal-AI.

**Objetivo:** Entrenar modelos YOLO independientes para LISA y Bosch, evaluar métricas (Precision, Recall, mAP@50) y guardar pesos optimizados.

**Hardware soportado:**
- GPU NVIDIA con CUDA (recomendado)
- CPU (fallback automático)
- Google Colab (con GPU Tesla T4)

**Ejecución:**
1. Ejecuta las celdas en orden
2. Configura parámetros en la sección 'Configuración'
3. No inicia entrenamiento largo sin revisar primero GPU/CPU disponible
4. Los pesos y métricas se guardan en rutas reproducibles

## 1. Instalación de dependencias

In [ ]:
# Detectar entorno (local vs Colab)
import sys
import os

IS_COLAB = 'google.colab' in sys.modules
print(f'Entorno: {"Google Colab" if IS_COLAB else "Local"}')

In [ ]:
# Instalar dependencias
import subprocess

packages = [
    'ultralytics>=8.0.0',
    'torch>=2.0.0',
    'torchvision>=0.15.0',
    'opencv-python>=4.8.0',
    'numpy>=1.24.0',
    'pandas>=2.0.0',
    'matplotlib>=3.7.0',
    'Pillow>=10.0.0'
]

for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print('✓ Dependencias instaladas')

## 2. Detección de hardware y configuración de dispositivo

In [ ]:
import torch
import platform
import psutil

# Información del sistema
print('=== INFORMACIÓN DEL SISTEMA ===')print(f'Sistema operativo: {platform.system()} {platform.release()}')
print(f'Procesador: {platform.processor()}')
print(f'RAM disponible: {psutil.virtual_memory().available / (1024**3):.2f} GB')

# Información de PyTorch y CUDA
print('\n=== PYTORCH Y CUDA ===')print(f'PyTorch versión: {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'GPU detectada: {torch.cuda.get_device_name(0)}')
    print(f'Memoria GPU: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')
    device = 'cuda'
    print(f'\n✓ Usando GPU para entrenamiento')
else:
    device = 'cpu'
    print(f'\n⚠ GPU no disponible. Usando CPU (más lento)')

print(f'\nDispositivo seleccionado: {device.upper()}')

## 3. Configuración de parámetros

In [ ]:
from pathlib import Path

# ============ CONFIGURACIÓN EDITABLE ============

# Rutas base
if IS_COLAB:
    # En Colab, montar Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/optisignal-ai')
else:
    # Local: asumir que el notebook está en notebooks/
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

DATA_ROOT = PROJECT_ROOT / 'data' / 'processed'
MODELS_ROOT = PROJECT_ROOT / 'models'
RESULTS_ROOT = PROJECT_ROOT / 'results' / 'phase_4'

# Crear directorios si no existen
MODELS_ROOT.mkdir(parents=True, exist_ok=True)
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

# Parámetros de entrenamiento
YOLO_MODEL = 'yolov8n'  # nano para inferencia rápida
EPOCHS = 5  # Cambiar a 50-100 para entrenamiento real
BATCH_SIZE = 16  # Reducir si hay problemas de memoria
IMG_SIZE = 640
PATIENCE = 10  # Early stopping
SEED = 42

# Datasets a entrenar
DATASETS = ['lisa', 'bosch']  # Entrenar ambos independientemente

print('=== CONFIGURACIÓN ===')print(f'Raíz del proyecto: {PROJECT_ROOT}')
print(f'Datos procesados: {DATA_ROOT}')
print(f'Modelos: {MODELS_ROOT}')
print(f'Resultados: {RESULTS_ROOT}')
print(f'\nModelo YOLO: {YOLO_MODEL}')
print(f'Épocas: {EPOCHS}')
print(f'Batch size: {BATCH_SIZE}')
print(f'Tamaño imagen: {IMG_SIZE}x{IMG_SIZE}')
print(f'Datasets: {DATASETS}')

## 4. Validación de estructura de datos

In [ ]:
import json

print('=== VALIDACIÓN DE DATASETS ===')
for dataset_name in DATASETS:
    dataset_path = DATA_ROOT / dataset_name / 'partitioned'
    
    if not dataset_path.exists():
        print(f'\n✗ {dataset_name.upper()}: No encontrado en {dataset_path}')
        continue
    
    # Verificar estructura
    required_dirs = ['images/train', 'images/val', 'images/test', 'labels/train', 'labels/val', 'labels/test']
    missing = [d for d in required_dirs if not (dataset_path / d).exists()]
    
    if missing:
        print(f'\n✗ {dataset_name.upper()}: Directorios faltantes: {missing}')
        continue
    
    # Contar imágenes
    train_count = len(list((dataset_path / 'images/train').glob('*.[jp][pn]g')))
    val_count = len(list((dataset_path / 'images/val').glob('*.[jp][pn]g')))
    test_count = len(list((dataset_path / 'images/test').glob('*.[jp][pn]g')))
    total = train_count + val_count + test_count
    
    # Leer manifiesto
    manifest_path = dataset_path / 'partition_manifest.json'
    if manifest_path.exists():
        with open(manifest_path) as f:
            manifest = json.load(f)
        print(f'\n✓ {dataset_name.upper()}')
        print(f'  Total imágenes: {total}')
        print(f'  Train: {train_count} ({train_count/total*100:.1f}%)')
        print(f'  Val:   {val_count} ({val_count/total*100:.1f}%)')
        print(f'  Test:  {test_count} ({test_count/total*100:.1f}%)')
        print(f'  Seed: {manifest.get("seed", "N/A")}')
    else:
        print(f'\n⚠ {dataset_name.upper()}: Manifiesto no encontrado')
        print(f'  Total imágenes: {total}')
        print(f'  Train: {train_count}, Val: {val_count}, Test: {test_count}')

## 5. Entrenamiento del modelo YOLO

In [ ]:
from ultralytics import YOLO
import torch

# Configurar seed para reproducibilidad
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

training_results = {}

for dataset_name in DATASETS:
    dataset_path = DATA_ROOT / dataset_name / 'partitioned'
    data_yaml = dataset_path / 'data.yaml'
    
    if not data_yaml.exists():
        print(f'\n✗ {dataset_name.upper()}: data.yaml no encontrado')
        continue
    
    print(f'\n=== ENTRENAMIENTO {dataset_name.upper()} ===')    print(f'Configuración:')
    print(f'  Modelo: {YOLO_MODEL}')
    print(f'  Épocas: {EPOCHS}')
    print(f'  Batch size: {BATCH_SIZE}')
    print(f'  Dispositivo: {device.upper()}')
    print(f'  Data YAML: {data_yaml}')
    
    # Cargar modelo preentrenado
    model = YOLO(f'{YOLO_MODEL}.pt')
    
    # Entrenar
    results = model.train(
        data=str(data_yaml),
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        device=device,
        patience=PATIENCE,
        seed=SEED,
        project=str(RESULTS_ROOT / dataset_name),
        name='train',
        exist_ok=True,
        verbose=True,
        save=True,
        save_period=5
    )
    
    # Guardar pesos
    best_weights = RESULTS_ROOT / dataset_name / 'train' / 'weights' / 'best.pt'
    if best_weights.exists():
        model_output = MODELS_ROOT / f'{dataset_name}_best.pt'
        import shutil
        shutil.copy2(best_weights, model_output)
        print(f'\n✓ Pesos guardados en {model_output}')
    
    training_results[dataset_name] = {
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'device': device,
        'results_path': str(RESULTS_ROOT / dataset_name / 'train')
    }
    
    print(f'\n✓ Entrenamiento {dataset_name.upper()} completado')

## 6. Evaluación del modelo

In [ ]:
import json

evaluation_results = {}

for dataset_name in DATASETS:
    model_path = MODELS_ROOT / f'{dataset_name}_best.pt'
    dataset_path = DATA_ROOT / dataset_name / 'partitioned'
    data_yaml = dataset_path / 'data.yaml'
    
    if not model_path.exists():
        print(f'\n⚠ {dataset_name.upper()}: Modelo no encontrado en {model_path}')
        continue
    
    print(f'\n=== EVALUACIÓN {dataset_name.upper()} ===')    
    # Cargar modelo entrenado
    model = YOLO(str(model_path))
    
    # Evaluar en conjunto de prueba
    metrics = model.val(data=str(data_yaml), device=device)
    
    # Extraer métricas principales
    eval_data = {
        'dataset': dataset_name,
        'model': YOLO_MODEL,
        'device': device,
        'metrics': {
            'precision': float(metrics.box.mp) if hasattr(metrics, 'box') else None,
            'recall': float(metrics.box.mr) if hasattr(metrics, 'box') else None,
            'mAP50': float(metrics.box.map50) if hasattr(metrics, 'box') else None,
            'mAP50_95': float(metrics.box.map) if hasattr(metrics, 'box') else None
        }
    }
    
    evaluation_results[dataset_name] = eval_data
    
    print(f'Precision: {eval_data["metrics"]["precision"]:.4f}' if eval_data['metrics']['precision'] else 'Precision: N/A')
    print(f'Recall: {eval_data["metrics"]["recall"]:.4f}' if eval_data['metrics']['recall'] else 'Recall: N/A')
    print(f'mAP@50: {eval_data["metrics"]["mAP50"]:.4f}' if eval_data['metrics']['mAP50'] else 'mAP@50: N/A')
    print(f'mAP@50-95: {eval_data["metrics"]["mAP50_95"]:.4f}' if eval_data['metrics']['mAP50_95'] else 'mAP@50-95: N/A')
    
    # Guardar resultados
    results_file = RESULTS_ROOT / f'{dataset_name}_evaluation.json'
    with open(results_file, 'w') as f:
        json.dump(eval_data, f, indent=2)
    print(f'\n✓ Resultados guardados en {results_file}')

## 7. Visualización de resultados

In [ ]:
import matplotlib.pyplot as plt
import json
from pathlib import Path

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Métricas de Entrenamiento YOLO - OptiSignal-AI', fontsize=16, fontweight='bold')

for idx, dataset_name in enumerate(DATASETS):
    results_dir = RESULTS_ROOT / dataset_name / 'train'
    
    if not results_dir.exists():
        print(f'⚠ {dataset_name.upper()}: Directorio de resultados no encontrado')
        continue
    
    # Buscar archivos de métricas
    results_csv = results_dir / 'results.csv'
    
    if results_csv.exists():
        import pandas as pd
        df = pd.read_csv(results_csv)
        
        # Gráfico 1: Pérdida
        ax = axes[0, idx]
        if 'train/loss' in df.columns and 'val/loss' in df.columns:
            ax.plot(df['train/loss'], label='Train Loss', linewidth=2)
            ax.plot(df['val/loss'], label='Val Loss', linewidth=2)
            ax.set_xlabel('Época')
            ax.set_ylabel('Pérdida')
            ax.set_title(f'{dataset_name.upper()} - Pérdida')
            ax.legend()
            ax.grid(True, alpha=0.3)
        
        # Gráfico 2: mAP
        ax = axes[1, idx]
        if 'metrics/mAP50(B)' in df.columns:
            ax.plot(df['metrics/mAP50(B)'], label='mAP@50', linewidth=2, color='green')
            ax.set_xlabel('Época')
            ax.set_ylabel('mAP@50')
            ax.set_title(f'{dataset_name.upper()} - mAP@50')
            ax.legend()
            ax.grid(True, alpha=0.3)
            ax.set_ylim([0, 1])
    else:
        ax = axes[0, idx]
        ax.text(0.5, 0.5, f'Sin datos de entrenamiento
para {dataset_name.upper()}', 
                ha='center', va='center', fontsize=12)
        ax.set_xticks([])
        ax.set_yticks([])

plt.tight_layout()
plt.savefig(RESULTS_ROOT / 'training_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n✓ Gráficos guardados en {RESULTS_ROOT / "training_metrics.png"}')

## 8. Prueba de inferencia

In [ ]:
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

print('=== PRUEBA DE INFERENCIA ===')
for dataset_name in DATASETS:
    model_path = MODELS_ROOT / f'{dataset_name}_best.pt'
    dataset_path = DATA_ROOT / dataset_name / 'partitioned'
    test_images_dir = dataset_path / 'images' / 'test'
    
    if not model_path.exists():
        print(f'\n⚠ {dataset_name.upper()}: Modelo no encontrado')
        continue
    
    if not test_images_dir.exists():
        print(f'\n⚠ {dataset_name.upper()}: Directorio de prueba no encontrado')
        continue
    
    # Obtener primeras 3 imágenes de prueba
    test_images = list(test_images_dir.glob('*.[jp][pn]g'))[:3]
    
    if not test_images:
        print(f'\n⚠ {dataset_name.upper()}: No hay imágenes de prueba')
        continue
    
    print(f'\n✓ {dataset_name.upper()}: Ejecutando inferencia en {len(test_images)} imágenes')
    
    model = YOLO(str(model_path))
    
    fig, axes = plt.subplots(1, len(test_images), figsize=(15, 5))
    if len(test_images) == 1:
        axes = [axes]
    
    for ax, img_path in zip(axes, test_images):
        # Inferencia
        results = model.predict(source=str(img_path), conf=0.5, device=device)
        
        # Leer imagen original
        img = cv2.imread(str(img_path))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # Dibujar detecciones
        if results[0].boxes is not None:
            for box in results[0].boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                conf = float(box.conf[0])
                cv2.rectangle(img_rgb, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(img_rgb, f'{conf:.2f}', (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        
        ax.imshow(img_rgb)
        ax.set_title(f'{img_path.name}\n({len(results[0].boxes) if results[0].boxes else 0} detecciones)')
        ax.axis('off')
    
    plt.suptitle(f'Inferencia - {dataset_name.upper()}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(RESULTS_ROOT / f'{dataset_name}_inference_samples.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f'✓ Muestras de inferencia guardadas')

## 9. Resumen y próximos pasos

In [ ]:
import json

print('\n' + '='*70)
print('RESUMEN DE FASE 4 - ENTRENAMIENTO YOLO')
print('='*70)

print(f'\nConfiguraciones utilizadas:')
print(f'  Modelo: {YOLO_MODEL}')
print(f'  Épocas: {EPOCHS}')
print(f'  Batch size: {BATCH_SIZE}')
print(f'  Dispositivo: {device.upper()}')
print(f'  Seed: {SEED}')

print(f'Archivos generados:')
for dataset_name in DATASETS:
    model_path = MODELS_ROOT / f'{dataset_name}_best.pt'
    if model_path.exists():
        size_mb = model_path.stat().st_size / (1024**2)
        print(f'  ✓ {model_path.name} ({size_mb:.1f} MB)')
    
    eval_file = RESULTS_ROOT / f'{dataset_name}_evaluation.json'
    if eval_file.exists():
        print(f'  ✓ {eval_file.name}')

print(f'\nDirectorio de resultados: {RESULTS_ROOT}')

print(f'\nPróximos pasos:')
print(f'  1. Revisar métricas en {RESULTS_ROOT}')
print(f'  2. Ajustar hiperparámetros si es necesario')
print(f'  3. Proceder a Fase 5: Clasificación de colores (HSV)')
print(f'  4. Integrar con máquina de estados (Fase 6)')

print('='*70)